Parte previa, git-github

1. Flujo de Trabajo en Git (MLOps Básico)
Primero, inicializar repositorio y gestionar las ramas como pide la consigna:

Paso 1: Configuración Inicial de Git y GitHub

Bash
git config --global user.name "Tu Nombre" en este casó usé mi cuenta de gmail donde también tengo colab pro
git config --global user.email "tu@email.com"
Luego creé una cuenta de github.

Paso 2: Inicializar el Proyecto (Rama master)
Convertimos la carpeta actual en un repositorio de Git.

En la terminal, dentro de tu carpeta:

Bash
git init
# Crear una rama principal limpia
git checkout -b master
Conectar con GitHub: Copiamos la URL del repositorio de GitHub 

git remote add origin https://github.com/screamingvalkirias-png/tarea-redes-neuronales.git

Tuve que pegar esto cd "3er Trimestre/Redes Neuronales/TAREA1" en el bash para ubicar la tarea. Luego cambio la rama etapa1:
git checkout -b etapa1
Luego
git push origin etapa1


Parte de encoder

Paso 1: Carga y Preprocesamiento de Datos

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# 1. Cargar el dataset (no necesitamos las etiquetas 'y' por ahora)
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

# 2. Normalización: Pasamos los píxeles de [0, 255] a [0, 1]
# Esto ayuda a que la red neuronal converja más rápido.
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print(f"Forma de los datos: {x_train.shape}") # Debería ser (60000, 28, 28)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Forma de los datos: (60000, 28, 28)


Paso 2: Arquitectura del Autoencoder
Divido la red en dos partes. El Encoder volveremos a usar en la Etapa 2 de la tarea.

In [2]:
# Definir la dimensión del vector latente (la compresión)
# Un valor de 64 significa que comprimiremos 784 píxeles en solo 64 valores.
latent_dim = 64 

# --- ENCODER ---
encoder_input = layers.Input(shape=(28, 28))
x = layers.Flatten()(encoder_input)
latent_vector = layers.Dense(latent_dim, activation='relu', name="vector_latente")(x)
encoder = models.Model(encoder_input, latent_vector, name="Encoder")

# --- DECODER ---
decoder_input = layers.Input(shape=(latent_dim,))
x = layers.Dense(784, activation='sigmoid')(decoder_input)
decoder_output = layers.Reshape((28, 28))(x)
decoder = models.Model(decoder_input, decoder_output, name="Decoder")

# --- AUTOENCODER COMPLETO ---
autoencoder_output = decoder(encoder(encoder_input))
autoencoder = models.Model(encoder_input, autoencoder_output, name="Autoencoder_Completo")

autoencoder.compile(optimizer='adam', loss='mean_squared_error')
autoencoder.summary()

Model: "Autoencoder_Completo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Encoder (Functional)            │ (None, 64)             │        50,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Decoder (Functional)            │ (None, 28, 28)         │        50,960 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,200 (395.31 KB)

 Trainable params: 101,200 (395.31 KB)

 Non-trainable params: 0 (0.00 B)

Paso 3: Entrenamiento y Guardado
Entreno el modelo para que la salida sea lo más parecida posible a la entrada.

In [8]:
# Entrenamos: fíjate que x_train es tanto la entrada como la salida esperada
history = autoencoder.fit(
    x_train, x_train,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test, x_test)
)

# GUARDAR EL ENCODER: Esto es vital para la Etapa 2

from google.colab import drive
drive.mount('/content/drive')
# Guardamos solo la parte que comprime los datos.
ruta_final = '/content/drive/MyDrive/Maestría en Ciencias de la Inteligencia Artificial/3er Trimestre/Redes Neuronales/TAREA1/encoder_fashion_mnist.keras'

# 3. Guardar
encoder.save(ruta_final)
print(f"¡Listo! Ya podés ver el archivo en tu disco G: de la PC")



Mounted at /content/drive
¡Listo! Ya podés ver el archivo en tu disco G: de la PC


Paso 4: MLOps - Guardar cambios en Git
Busco que el código corra y ver el loss bajar, ahí se registra el avance en la rama etapa1:

En la terminal ejecuto:

 1. Agregamos solo el notebook

cd "3er Trimestre/Redes Neuronales/TAREA1"

ls (pego esto para saber si está ahí)

git add PARTE1.ipynb

2. Agregamos solo el modelo (el nuevo .keras)

git add encoder_fashion_mnist.keras

3. Guardamos los cambios localmente

git commit -m "Etapa 1: Encoder guardado vía Cloud-Sync"

4. Subimos a GitHub (asegurándonos de que sea la rama etapa1)

git push origin etapa1

antes de guardar el modelo. me di cuenta de que en mi carpeta no estaba el archivo guardado del encoder. así que

In [5]:
import os
print("El notebook cree que está en:", os.getcwd())
print("Archivos en esta carpeta:", os.listdir())


El notebook cree que está en: /content
Archivos en esta carpeta: ['.config', 'encoder_fashion_mnist.h5', 'encoder_fashion_mnist.keras', 'sample_data']


consultando, hice modificaciones y me salió esto:

Para guardar el archivo exactamente en esa carpeta de tu maestría desde Colab, tenés que usar este bloque. Solo recordá que sin la primera línea (el mount), el código no tiene permiso para entrar a tu Drive, por más que pongas la ruta correcta.

Python
1. El "puente" (Es obligatorio para que Colab vea tus carpetas)
from google.colab import drive
drive.mount('/content/drive')

 2. La ruta modificada (Colab usa / y no entiende el disco G:)
Ojo: Agregué la barra que faltaba entre TAREA1 y el nombre del archivo
ruta_final = '/content/drive/MyDrive/Maestría en Ciencias de la Inteligencia Artificial/3er Trimestre/Redes Neuronales/TAREA1/encoder_fashion_mnist.keras'

3. Guardar
encoder.save(ruta_final)
print(f"¡Listo! Ya podés ver el archivo en tu disco G: de la PC")